#  Agentic RAG with OpenAI Function Calling

## Learning Objectives

By the end of this notebook, you will understand:

1. **The difference between RAG and Agentic RAG**
   - Plain RAG: Always retrieves, then generates
   - Agentic RAG: Model decides when to retrieve

2. **OpenAI Function Calling mechanics**
   - Defining tool schemas in JSON format
   - Multi-turn conversation patterns
   - Parsing and executing function calls

3. **Conversation history management**
   - Why LLMs are stateless (no memory between calls)
   - Building message history with user inputs, function calls, and results
   - The role of `call_id` in linking function results

4. **Cost implications**
   - Token usage tracking
   - Calculating costs for gpt-5.4-mini
   - Why agentic workflows cost 2x-3x more than plain RAG

## Notebook Structure

This notebook is organized into 4 main parts:

### Part 0: Understanding RAG vs Agentic RAG
- Visual comparison of both architectures
- When to use each pattern

### Part 1: Asking Without Tools (Baseline)
- See what happens without function calling
- Understand the limitations

### Part 2: Function Calling Setup
- Define the Python search function
- Create the JSON tool schema
- Call the model with tools available

### Part 3: The Function Calling Loop
- Extract and parse function calls
- Execute the search
- Send results back to the model
- Get the final answer

### Part 4: Token Usage & Cost Analysis
- Track input/output tokens
- Calculate API costs
- Understand agentic workflow economics

Let's get started! 🚀

# Agentic RAG Study Notes: Building an AI Assistant with Function Calling

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system with agentic capabilities using OpenAI's function calling feature.

The mechanism that makes this possible is **function calling**.

## The Core Problem: RAG vs Agentic RAG

- **Agent**: LLM decides. It chooses which actions to take and when to stop.

### Traditional RAG Flow (Fixed Pipeline)- **RAG**: Developer decides. We fix the steps up front, so search always runs once with the exact user query.

In traditional RAG, the developer decides the execution flow:

### Key Difference



```text
The LLM searched, saw the results were bad, and **decided on its own** to try again with a different query. We didn't write any code to handle typos.

User: How do I run Olama? (typo)

  ↓```

search("Olama") → no useful results

LLM: "Here's how to run Ollama locally..."

  ↓  ↓

LLM: "I don't have information about Olama."search("Ollama") → found results!

```  ↓

LLM: "Hmm, no results. Maybe a typo for 'Ollama'?"

The LLM is a **passenger**, not a driver. It never sees the bad search results, so it can't try again with a corrected query.  ↓

search("Olama") → no useful results

### Agentic RAG Flow (LLM Decides)  ↓

With an agent, the LLM is in charge and makes decisions:LLM: "I'll search for 'Olama'"

  ↓

```User: How do I run Olama?

## Step 1: Initialize OpenAI Client

First, we load environment variables and create an OpenAI client. This client will handle all API calls to OpenAI's models, including our model that will have access to the search tool function.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

## Step 2: Load FAQ Data and Build Search Index

We load course FAQ data from an external source and build a full-text search index using the `minsearch` library. This index will be used by the RAG system to quickly retrieve relevant Q&A pairs. The index is configured to search across question, section, and answer fields with boosted weights for question relevance.

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

## Step 3: Create RAG Assistant

Instantiate the `RAGBase` class which encapsulates our RAG system. This assistant will combine the search index with the OpenAI LLM to answer questions by first retrieving relevant context from the FAQ database, then using that context to generate accurate responses.

In [51]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

## Step 4: Test Basic RAG Query

Let's test the RAG assistant with a straightforward question to see how it retrieves and uses FAQ data.

In [53]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download) for your operating system:
   - **macOS**: download the `.pkg` and install it
   - **Windows**: download the `.msi` and install it
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start Ollama locally** by opening a terminal and running:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. **Test the local server** with:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you get a connection refused error while prompting Ollama, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


## Step 5: Test with Intentional Misspelling

Deliberately misspelling "Ollama" as "Olama" to test if our RAG system can handle spelling variations and still find relevant information.

This demonstrates the limitation of fixed RAG pipelines. The solution? **Agentic RAG** where the LLM can retry with corrected queries.

**Expected Result**: The traditional RAG pipeline will fail because:

1. It searches for the exact query "Olama" (typo)3. The LLM receives empty context and cannot provide a useful answer
2. No documents match this misspelled term

In [54]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The FAQ doesn’t mention **Olama** specifically.

If you mean running the course locally, the FAQ says you can do that instead of using Codespaces if you’re comfortable setting up the needed tools yourself: **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. If you run locally, you should **document your setup** and keep it **reproducible**.


## 🔧 Part 2: Function Calling - Defining the Search Function

Before we can give the model a tool, we need to define the actual Python function that will execute when the tool is called.

### The Python Function

We create a top-level `search()` function that queries the index directly. The model will reference it by this name, so we keep the Python function name and the tool name aligned to make dispatch easier later.

```python
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
```

This function:
- Takes a `query` string parameter
- Configures search boosting (questions weighted 3x, sections 0.5x)
- Filters results to only the "llm-zoomcamp" course
- Returns the top 5 matching FAQ entries

**Important**: The model doesn't see our Python code. We'll describe this function in JSON schema format next.

In [55]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—if the course is still open, you can usually join it.\n\nTo confirm, I’d need a bit more context:\n- Which course is it?\n- Is it currently in progress or already finished?\n- Are you asking about enrollment, access, or catching up on missed material?\n\nIf you want, send me the course name or link and I can help you figure out the next step.'

## 🔎 Part 1: Asking Without Tools (Baseline)

Before we add function calling, let's see what happens when we ask the question **without** giving the model access to the search tool.

### The Setup

We create a simple conversation with two messages:
1. **System message**: Sets context ("You're a course teaching assistant")
2. **User message**: Asks the actual question

```python
messages = [
    {"role": "system", "content": "You're a course teaching assistant..."},
    {"role": "user", "content": question}
]
```

Then we call the OpenAI API without any tools:

```python
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    # Note: no 'tools' parameter here
)
```

### What to Expect

Without the FAQ search tool, the model can only:
- Use its general training knowledge
- Make educated guesses
- Provide generic advice

It **cannot**:
- Search the course FAQ database
- Reference specific course policies
- Give course-specific deadlines or procedures

**Result**: The answer might be reasonable but won't be grounded in the actual course materials. This is why we need tools!

In [56]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

## Step 7: Define Search Tool Schema

Now we define the **tool schema** that describes our search function to the model in JSON format.

### The Tool Schema Structure

```json
{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search course FAQ database...",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"],
            "additionalProperties": false
        }
    }
}
```

### Breaking Down the Schema

1. **`type: "function"`** - Declares this is a function tool
2. **`function.name`** - Must match the Python function name (`search`)
3. **`function.description`** - Tells the model what this function does and when to use it
4. **`parameters.type: "object"`** - Arguments are passed as a JSON object
5. **`properties.query`** - The `query` parameter is a string
6. **`required: ["query"]`** - The model must provide this parameter
7. **`additionalProperties: false`** - Model cannot add unexpected parameters

### Why This Matters

The model uses this schema to:
- **Decide** when to call the function (based on description)
- **Generate** valid JSON arguments matching the schema
- **Validate** its own function call structure

**Key Point**: The model doesn't see your Python code. It only sees this JSON description. Make the description clear!

## 🔄 Part 3: The Function Calling Loop

### Step 10: Executing the Function and Sending Results Back

The function call from the model contains JSON arguments. Here's what we do:

1. **Parse the arguments** from the JSON string
2. **Call our Python search function** with those arguments  
3. **Serialize the result** back to JSON
4. **Send it back** to the model with the proper `call_id` linking

```python
import json

# 1. Extract the function call
call = response.output[0]

# 2. Parse JSON arguments
args = json.loads(call.arguments)

# 3. Execute our Python function
results = search(**args)

# 4. Convert results to JSON
result_json = json.dumps(results, indent=2)
```

### Step 11: Adding Results to Conversation History

Now we update the conversation history. This is **critical** because LLMs are stateless between API calls.

```python
# Add the model's function call to history
messages.extend(response.output)

# Add our function result
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,  # Links to the specific function call
    "output": result_json,
})
```

**Why the `call_id`?** If the model makes multiple function calls in one turn, each gets its own `call_id`. This links each result to the correct call.

**Why extend the history?** We have to send the whole history because LLMs are stateless. The memory is the list you send as `input`. If you send only the tool result, the model has no idea what's going on.

### Step 12: Second API Call - Getting the Final Answer

We call the API a second time with the expanded history:

```python
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,  # Now contains: question + function call + results
    tools=[search_tool],
)

response.output_text
```

This time the model has:
1. The original question
2. Its own decision to call `search`
3. The FAQ results from our function

Now it can produce a proper course-specific answer!

**Cost Consideration**: With plain RAG we made one call. With agentic RAG we make two or more calls. Turning RAG agentic means more round-trips and higher costs.

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

## Step 8: Call Model with Tools Available

Now we send the **same question** as before, but this time we include the tool in the request via the `tools` parameter.

**Query Rewriting**: Look at the arguments. The model doesn't pass our question verbatim. It judges the raw question isn't the best query to search with. So it rewrites our enrollment question into search keywords like "enroll late join course".

**What to Expect**: Instead of a message with the answer, the response will contain a `function_call` entry. The model will **decide** it needs to search the FAQ before answering. Rather than reply directly, it will ask us to run the search function first.

In [18]:
len(response.output)

1

## Step 9: Check Response Length

See how many output items the model returned.

**Expected**: The response contains function call(s) instead of a text answer. The model decided to use the tool before responding.

In [ ]:
# Step 10: Extract Tool Call
# Examine the function call that the model generated.
response.output

In [ ]:
call = response.output[0]
call

In [ ]:
# Step 11: Parse Function Arguments
# Extract and parse the JSON arguments from the tool call.
import json

args
args = json.loads(call.arguments)

{'query': 'join the course late discovered can I join it now enrollment access late registration'}

In [ ]:
import json

args = json.loads(call.arguments)
args

## Step 12: Verify Function Name

Check which function the model wants to call.

In [ ]:
call.name

## Step 13: Execute Search Function

Execute the actual search using the assistant's search method with the extracted arguments.

In [ ]:
results = assistant.search(**args)

## Step 14: Format Results as JSON

Convert search results to JSON string for sending back to the model.

In [ ]:
result_json = json.dumps(results, indent=2)

## Step 15: Create Function Output Message

Package the search results as a function call output following OpenAI's format.

In [ ]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

## Step 16: Add Tool Call to Messages

Add the function call to conversation history.

In [ ]:
messages.append(call)

## Step 17: Add Function Output to Messages

Add the function results to conversation history.

In [ ]:
messages.append(function_call_output)

## Step 18: View Complete Conversation

Inspect the full conversation history with all messages.

In [ ]:
messages

## Step 19: Final API Call with Context

Send the complete conversation back to the model so it can generate a final answer using the search results.

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

## Step 20: Display Final Answer

Print the model's final response grounded in FAQ data.

In [ ]:
print(response.output_text)

## Step 21: Monitor Token Usage

Track the computational cost of this agentic interaction.

In [ ]:
usage = response.usage
usage.input_tokens, usage.output_tokens

## 💰 Part 4: Token Usage and Cost Calculation

### Understanding Token Consumption in Agentic RAG

Every API call to OpenAI returns a `usage` object with two key metrics:

```python
usage = response.usage
print(f"Input tokens: {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
```

- **Input tokens**: Everything we send to the model (messages, tool schemas, function results)
- **Output tokens**: Everything the model generates (function calls, final text responses)

### Cost Calculation for gpt-5.4-mini

OpenAI pricing for `gpt-5.4-mini`:
- Input: $0.15 per million tokens
- Output: $0.60 per million tokens

Here's a helper function to calculate costs:

```python
def calculate_gpt54mini_price(usage):
    input_cost_per_1M = 0.15   # $0.15 per 1M input tokens
    output_cost_per_1M = 0.60  # $0.60 per 1M output tokens
    
    input_cost = (usage.input_tokens / 1_000_000) * input_cost_per_1M
    output_cost = (usage.output_tokens / 1_000_000) * output_cost_per_1M
    
    total_cost = input_cost + output_cost
    
    return {
        'input_tokens': usage.input_tokens,
        'output_tokens': usage.output_tokens,
        'input_cost': round(input_cost, 6),
        'output_cost': round(output_cost, 6),
        'total_cost': round(total_cost, 6)
    }
```

### Why Agentic RAG Costs More

**Plain RAG (1 API call)**:
- User question → Retrieve context → LLM call → Answer
- Cost: ~500-1000 input tokens + 100-200 output tokens

**Agentic RAG (2+ API calls)**:
- Call 1: Question + tool schema → Function call decision
- Call 2: Question + tool schema + function call + results → Final answer
- Cost: 2x-3x more tokens because we send the full history each time

**Trade-off**: Higher cost but better accuracy and flexibility. The model can rewrite queries, decide when retrieval is needed, and handle multi-step reasoning.

### Tracking Costs Across Multi-Turn Conversations

For complete cost tracking in agentic workflows, sum up usage from all API calls:

```python
total_input_tokens = 0
total_output_tokens = 0

# After each API call:
total_input_tokens += response.usage.input_tokens
total_output_tokens += response.usage.output_tokens

# Calculate final cost
final_cost = calculate_gpt54mini_price(
    type('Usage', (), {
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    })()
)
```

In [ ]:
def calculate_gpt54mini_price(usage):
    """Calculate cost for gpt-5.4-mini API usage."""
    input_cost_per_1M = 0.15   # $0.15 per 1M input tokens
    output_cost_per_1M = 0.60  # $0.60 per 1M output tokens
    
    input_cost = (usage.input_tokens / 1_000_000) * input_cost_per_1M
    output_cost = (usage.output_tokens / 1_000_000) * output_cost_per_1M
    
    total_cost = input_cost + output_cost
    
    return {
        'input_tokens': usage.input_tokens,
        'output_tokens': usage.output_tokens,
        'input_cost': round(input_cost, 6),
        'output_cost': round(output_cost, 6),
        'total_cost': round(total_cost, 6)
    }

# Calculate cost for this interaction
cost_breakdown = calculate_gpt54mini_price(usage)
print(f"Input tokens: {cost_breakdown['input_tokens']}")
print(f"Output tokens: {cost_breakdown['output_tokens']}")
print(f"Input cost: ${cost_breakdown['input_cost']}")
print(f"Output cost: ${cost_breakdown['output_cost']}")
print(f"Total cost: ${cost_breakdown['total_cost']}")

## 🎯 Key Takeaways: Agentic RAG Architecture

### What We Learned

1. **Function Calling Fundamentals**
   - Models can autonomously decide when and how to use tools
   - Tool schemas describe functions to the model in JSON format
   - The model generates valid function calls based on the schema

2. **Multi-turn Conversation Pattern**
   - User question → Model function call → Execute function → Send results → Final answer
   - LLMs are stateless: we must send the full conversation history each time
   - `call_id` links function results to specific function calls

3. **RAG vs Agentic RAG**
   - Plain RAG: Always retrieves context before answering
   - Agentic RAG: Model decides if retrieval is needed
   - Agentic pattern enables query rewriting and multi-step reasoning

4. **Cost & Token Management**
   - Track `usage.input_tokens` and `usage.output_tokens` for each call
   - Agentic workflows cost 2x-3x more due to multiple API calls
   - Trade-off: higher cost for better accuracy and flexibility

### Next Steps

- **Add more tools**: Combine search with calculations, API calls, or database queries
- **Handle errors**: Add try/except blocks around function execution
- **Implement streaming**: Use streaming responses for better UX
- **Add conversation memory**: Store conversation history in a database for persistence
- **Optimize costs**: Use cheaper models for function calling, expensive models for final answers

You now have the foundation to build production-grade agentic RAG systems! 🚀

## 🔄 Adapting This Pattern to Other Use Cases

The function calling pattern you learned here is universal. Here's how to adapt it:

### Example: Power BI Report Search

Instead of searching FAQs, imagine searching Power BI reports:

```python
def search_powerbi_reports(query, workspace=None):
    """Search Power BI reports by name or description."""
    # Your Power BI API logic here
    return matching_reports

# Tool schema
powerbi_tool = {
    "type": "function",
    "function": {
        "name": "search_powerbi_reports",
        "description": "Search for Power BI reports by name or description. Use this when the user asks about reports, dashboards, or analytics.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query for report names or descriptions"
                },
                "workspace": {
                    "type": "string",
                    "description": "Optional: filter by workspace name"
                }
            },
            "required": ["query"],
            "additionalProperties": false
        }
    }
}
```

### Key Adaptation Points

1. **Function Implementation**: Change what the function does (API call, database query, calculation)
2. **Tool Schema**: Update `name`, `description`, and `parameters` to match your function
3. **Function Execution**: Parse `call.arguments` and call your function with `**args`
4. **Everything Else Stays the Same**: Message history, `call_id` linking, multi-turn pattern

### Multi-Tool Systems

You can provide multiple tools at once:

```python
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool, powerbi_tool, calculator_tool]
)
```

The model will:
- Choose the right tool for the job
- Or call multiple tools in sequence
- Or decide no tool is needed

This makes the system truly agentic!